# Config System Example

This notebook demonstrates the fluent `Config` builder in
`robotdataset.configuration_system`, used for configuring WorldModel /
VLA training runs and for generating W&B sweep configs.


In [30]:
from robotdataset.configuration_system import Config

## 1. Building a config with the fluent API

In [36]:
cfg = (
    Config().from_file("sample_config.yaml")
    .dataset(
        name="lalala"
    )
    .training(
        learning_rate=0.002
    )
    .set_bounds("training", "learning_rate", 0.001, 0.51)
)

In [25]:
type(cfg)

robotdataset.configuration_system.config.Config

In [ ]:
cfg.training.set_bounds("learning_rate", 0.001, 0.01)


Config
  dataset:
    name: 'lalala' (str)
    batch_size: 32 (int)
    shuffle: True (bool)
  training:
    learning_rate: float [bounds={'min': 0.001, 'max': 0.01}]
    optimizer: 'torch.optim.Adam' (str)
    num_epochs: 100 (int)

In [38]:
cfg

Config
  dataset:
    name: 'lalala' (str)
    batch_size: 32 (int)
    shuffle: True (bool)
  training:
    learning_rate: float [bounds={'min': 0.001, 'max': 0.51}]
    optimizer: 'torch.optim.Adam' (str)
    num_epochs: 100 (int)

Groups and fields are learned dynamically: calling `cfg.dataset(...)`
creates the `dataset` group (if it doesn't exist yet) and records each
field's value. Every field ends up with two required subfields, `type`
and `default`, which are inferred automatically from the plain value.


In [ ]:
# print(cfg.groups())
# print(cfg.fields("dataset"))
# print(cfg.schema("dataset", "batch_size"))


['dataset', 'training']
['name', 'batch_size', 'shuffle']
FieldSpec(name='batch_size', type='int', default=32, bounds=None, values=None)


## 2. Loading a config from YAML

In [3]:
from pathlib import Path

config_path = Path("../robotdataset/configuration_system/example_config.yaml")
print(config_path.read_text())


dataset:
  name: oxe
  batch_size: 32
  shuffle: true

training:
  learning_rate: 1.0e-4
  optimizer: torch.optim.Adam
  num_epochs: 100



Notice the file has **no** `type`/`bounds`/`values` info at all — just
plain values. That's fine: `type` and `default` are learned by
inference, and hyperparameter opt settings (`bounds`/`values`) are
attached afterward, not required in the file.


In [4]:
cfg = Config.from_file(config_path)
cfg


Config
  dataset:
    name: 'oxe' (str)
    batch_size: 32 (int)
    shuffle: True (bool)
  training:
    learning_rate: 0.0001 (float)
    optimizer: 'torch.optim.Adam' (str)
    num_epochs: 100 (int)

In [5]:
lr_spec = cfg.schema("training", "learning_rate")
opt_spec = cfg.schema("training", "optimizer")
print(lr_spec)
print(opt_spec)


FieldSpec(name='learning_rate', type='float', default=0.0001, bounds=None, values=None)
FieldSpec(name='optimizer', type='str', default='torch.optim.Adam', bounds=None, values=None)


## 3. Attaching hyperparameter opt settings after loading

In [6]:
cfg.set_bounds("training", "learning_rate", min=1e-5, max=1e-2)
cfg.training.set_values("optimizer", ["adam", "sgd", "adamw"])

cfg


Config
  dataset:
    name: 'oxe' (str)
    batch_size: 32 (int)
    shuffle: True (bool)
  training:
    learning_rate: float [bounds={'min': 1e-05, 'max': 0.01}]
    optimizer: str [values=['adam', 'sgd', 'adamw']]
    num_epochs: 100 (int)

`set_bounds`/`set_values` work on `Config` directly (`cfg.set_bounds(group, field, ...)`)
or on a group proxy (`cfg.training.set_values(field, ...)`). Either way, the
field's `default` from the file is preserved alongside its new bounds/values.


In [7]:
print(cfg.schema("training", "learning_rate"))
print(cfg.schema("training", "optimizer"))


FieldSpec(name='learning_rate', type='float', default=0.0001, bounds={'min': 1e-05, 'max': 0.01}, values=None)
FieldSpec(name='optimizer', type='str', default='torch.optim.Adam', bounds=None, values=['adam', 'sgd', 'adamw'])


## 4. Updating and saving a config

In [8]:
cfg.training(num_epochs=200)  # plain fields update the same way
cfg.save("/tmp/updated_config.yaml")
print(Path("/tmp/updated_config.yaml").read_text())


dataset:
  name: oxe
  batch_size: 32
  shuffle: true
training:
  learning_rate:
    type: float
    default: 0.0001
    bounds:
      min: 1.0e-05
      max: 0.01
  optimizer:
    type: str
    default: torch.optim.Adam
    values:
    - adam
    - sgd
    - adamw
  num_epochs: 200



## 5. Exporting a W&B sweep config

In [9]:
sweep_config = cfg.to_sweep(
    method="bayes",
    metric={"name": "val_loss", "goal": "minimize"},
)
sweep_config


{'method': 'bayes',
 'parameters': {'dataset.name': {'value': 'oxe'},
  'dataset.batch_size': {'value': 32},
  'dataset.shuffle': {'value': True},
  'training.learning_rate': {'min': 1e-05, 'max': 0.01},
  'training.optimizer': {'values': ['adam', 'sgd', 'adamw']},
  'training.num_epochs': {'value': 200}},
 'metric': {'name': 'val_loss', 'goal': 'minimize'}}

- Fields with `bounds` (e.g. `training.learning_rate`) become continuous
  `min`/`max` ranges.
- Fields with `values` (e.g. `training.optimizer`) become discrete/categorical
  choices.
- Every other field is exported as a fixed `value`, so the full config is
  still captured in the sweep.


In [10]:
cfg.to_sweep_file("/tmp/sweep.yaml", method="bayes")
print(Path("/tmp/sweep.yaml").read_text())


method: bayes
parameters:
  dataset.name:
    value: oxe
  dataset.batch_size:
    value: 32
  dataset.shuffle:
    value: true
  training.learning_rate:
    min: 1.0e-05
    max: 0.01
  training.optimizer:
    values:
    - adam
    - sgd
    - adamw
  training.num_epochs:
    value: 200



The resulting `sweep.yaml` can be passed directly to `wandb sweep`:

```bash
wandb sweep /tmp/sweep.yaml
```
